# Day 2: 信号交差点と Incremental Node Model

## 学習目標
- UXsimにおける信号制御 (`addNode(signal=[...])`, `addLink(signal_group=...)`) の仕組みを理解する
- 信号は「容量配分ロジック(Incremental Node Model)」とは独立した「通行可否のレイヤー」として実装されていることをコードで確認する
- 飽和交通流率・青時間配分・サイクル長と、リンクの実効容量の関係を数値で確認する
- 信号オフセットを変えることで系統(グリーンウェーブ)がどう効くかを観察する

## 前提知識
- [`docs/theory.md`](../docs/theory.md) 3節「Incremental Node Model」の復習
- 三角形基本図の `capacity`（`Day 1` で確認済み）


## 信号のしくみ（要点）

`W.addNode(name, x, y, signal=[g1, g2, ...])` の `signal` は各フェーズの青時間 [s] のリストです。
例えば `signal=[30, 30]` は 2 フェーズ・サイクル長 60 秒（黄色・全赤は考慮しない単純化モデル）を意味します。

`W.addLink(..., signal_group=0)` で、そのリンクがどのフェーズで青になるかを指定します。
`signal_group` が現在の `node.signal_phase` と一致する（またはリストに含まれる）リンクだけが
その瞬間 `Node.transfer()`（Incremental Node Model）の対象になります。

つまり信号は「容量配分の計算式」自体を変えるのではなく、「そのタイムステップで
配分の対象になれるリンクを絞り込むフィルタ」として実装されています。
これは合流の `merge_priority` と同じ土台（容量制約＋優先度）の上に、
信号は「時間的な通行可否」を追加している、というイメージです。

実効容量の目安式:

```
実効容量 ≈ capacity × (green_time / cycle_length)
```


In [ ]:
from uxsim import World

# 単純な十字交差点: 南北方向 (ns) と 東西方向 (ew) の2方向、それぞれ1本ずつの流入・流出リンク
W = World(
    name="signal_demo",
    deltan=5,
    tmax=3600,
    print_mode=1, save_mode=1, show_mode=1,
    random_seed=0,
)

# 交差点ノードに2フェーズ信号を設定: 南北30秒 → 東西30秒 → 繰り返し(サイクル長60秒)
W.addNode("north", 0, 1)
W.addNode("south", 0, -1)
W.addNode("east", 1, 0)
W.addNode("west", -1, 0)
W.addNode("intersection", 0, 0, signal=[30, 30])

# ns方向はsignal_group=0(フェーズ0で青)、ew方向はsignal_group=1(フェーズ1で青)
W.addLink("ns_in", "north", "intersection", length=500, free_flow_speed=15, number_of_lanes=1, signal_group=0)
W.addLink("ns_out", "intersection", "south", length=500, free_flow_speed=15, number_of_lanes=1)
W.addLink("ew_in", "east", "intersection", length=500, free_flow_speed=15, number_of_lanes=1, signal_group=1)
W.addLink("ew_out", "intersection", "west", length=500, free_flow_speed=15, number_of_lanes=1)

W.adddemand("north", "south", t_start=0, t_end=3000, flow=0.30)
W.adddemand("east", "west", t_start=0, t_end=3000, flow=0.30)

link_ns = W.get_link("ns_in")
print(f"ns_in の容量(青時間無限大の場合の理論上限): {link_ns.capacity:.3f} veh/s")
print(f"青時間比率 30/60 = 0.5 を掛けた実効容量の目安: {link_ns.capacity*0.5:.3f} veh/s")
print("→ 需要0.30 veh/sが、この実効容量を上回っていないか確認してから実行しよう")


In [ ]:
W.exec_simulation()
W.analyzer.print_simple_stats()

# ns方向・ew方向それぞれの実測スループット(完了台数/時間)をざっくり確認
df = W.analyzer.link_traffic_state_to_pandas()
for name in ["ns_in", "ew_in"]:
    q_mean = df[df["link"] == name]["q"].mean()
    print(f"{name}: 平均流率(実測) = {q_mean:.3f} veh/s")


## Part B: 演習

以下のTODOを埋めて、信号設計が交通処理性能にどう影響するかを自分の手で確認してください。

1. **不均等な青時間配分**: `signal=[30, 30]` を `signal=[40, 20]` に変更し、
   ns方向・ew方向の需要バランス（`flow`の値）をどう変えれば両方とも捌ききれるか調整してみましょう。
2. **系統(グリーンウェーブ)の実験**: `intersection` の下流にもう1つ信号付き交差点 `intersection2` を追加し、
   `signal_offset` を、上流の交差点からの走行時間（リンク長 / 自由流速度）に合わせて設定してください。
   時空間図 (`W.analyzer.time_space_diagram_traj_links`) で、車列が2つの信号を止まらずに
   通過できているかを確認しましょう。


In [ ]:
# TODO 1: 青時間配分を変えて実験する
# W_uneven = World(name="signal_uneven", deltan=5, tmax=3600, print_mode=1, save_mode=1, show_mode=1, random_seed=0)
# W_uneven.addNode("intersection", 0, 0, signal=[?, ?])  # ← ここを埋める
# ...(ns_in/ns_out/ew_in/ew_outとdemandを同様に定義)...
# W_uneven.exec_simulation()
# W_uneven.analyzer.print_simple_stats()


In [ ]:
# TODO 2: 下流にもう1つ信号交差点を追加し、signal_offsetでグリーンウェーブを作る
# 走行時間の目安: length / free_flow_speed
# offset = ?  # ← ここを埋める
# W.addNode("intersection2", 2, 0, signal=[30, 30], signal_offset=offset)
# ...


## Part C: 考察

- なぜUXsimは「信号」を独立したレイヤーとして実装しているのでしょうか？もし信号ロジックを
  `merge_priority` の中に押し込めようとすると、どんな不都合が起きそうか考えてみてください。
- 実務のオフセット設計（系統制御・グリーンウェーブ）と、今回の演習の対応関係を一文でまとめてください。
